In [9]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [10]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('YellowTaxi2024-10') \
    .getOrCreate()

### Question 1

In [11]:
spark.version

'3.5.5'

In [5]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-08 23:35:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.187, 18.155.128.222, 18.155.128.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  89.1MB/s    in 0.7s    

2025-03-08 23:35:35 (89.1 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [6]:
!ls -lh yellow_tripdata_2024-10.parquet

-rw-rw-r-- 1 xcluo xcluo 62M Dec 18 21:21 yellow_tripdata_2024-10.parquet


In [12]:
df = spark.read.parquet("yellow_tripdata_2024-10.parquet")

In [14]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [18]:
df = df.repartition(4)
df.write.parquet("yellow_2024_10_partitioned")

### Question 2

In [19]:
ls -lh yellow_2024_10_partitioned/

total 90M
-rw-r--r-- 1 xcluo xcluo   0 Mar  8 23:39 _SUCCESS
-rw-r--r-- 1 xcluo xcluo 23M Mar  8 23:39 part-00000-2cb50618-4df0-4054-a9de-4f2789e30c23-c000.snappy.parquet
-rw-r--r-- 1 xcluo xcluo 23M Mar  8 23:39 part-00001-2cb50618-4df0-4054-a9de-4f2789e30c23-c000.snappy.parquet
-rw-r--r-- 1 xcluo xcluo 23M Mar  8 23:39 part-00002-2cb50618-4df0-4054-a9de-4f2789e30c23-c000.snappy.parquet
-rw-r--r-- 1 xcluo xcluo 23M Mar  8 23:39 part-00003-2cb50618-4df0-4054-a9de-4f2789e30c23-c000.snappy.parquet


### Question 3

In [32]:
from pyspark.sql.functions import to_date
df_filtered = df.filter(to_date(df.tpep_pickup_datetime) == "2024-10-15")

In [33]:
df_filtered.count()

128893

### Question 4

In [38]:
from pyspark.sql.functions import col

df = df.withColumn("trip_duration_hours", \
                   (col("tpep_dropoff_datetime") - col("tpep_pickup_datetime")) / 3600)
longest_trip = df.selectExpr("MAX(trip_duration_hours) AS max_duration").collect()[0]["max_duration"]

In [39]:
longest_trip

datetime.timedelta(seconds=162, microseconds=617778)

In [40]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-08 23:55:00--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.6, 18.155.128.222, 18.155.128.46, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.6|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-08 23:55:00 (149 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [41]:
!ls -lh taxi_zone_lookup.csv

-rw-rw-r-- 1 xcluo xcluo 13K Feb 22  2024 taxi_zone_lookup.csv


### Question 6

In [42]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [43]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [44]:
df.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [45]:
result = spark.sql("""
    SELECT z.Zone, COUNT(t.PULocationID) AS pickup_count
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    LIMIT 1
""")

result.show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
+--------------------+------------+

